# Interface VAE detector

In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, average_precision_score
import logging

# ── Configuration ───────────────────────────────────────────────────────────────
CFG = {
    "backbone": "dinov2_vitl14", 
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "threshold": 0.5,  
    "image_size": 224,  
}

# ── Transforms ─────────────────────────────────────────────────────────────────
transform_224 = transforms.Compose([
    transforms.Resize((CFG["image_size"], CFG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ── Classe ImageFolderFlat ─────────────────
class ImageFolderFlat(torch.utils.data.Dataset):
    def __init__(self, root, transform=None, label=0):
        self.root = Path(root)
        self.transform = transform
        self.label = label
        self.paths = sorted([p for p in self.root.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.label

# ── Model DINOv2Detector ───────────────────────────────────────────────────────
class DINOv2Detector(nn.Module):
    def __init__(self, backbone_name: str):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name, pretrained=True)
        self.head = nn.Linear(self.backbone.embed_dim, 1)
        
        nn.init.xavier_normal_(self.head.weight)
        nn.init.constant_(self.head.bias, 0)

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

# ── Charger un modèle entraîné ─────────────────────────────────────────────────
def load_trained_model(backbone_name: str, checkpoint_path: str):
    """
    Charge un modèle DINOv2Detector avec les poids d'un checkpoint.
    """
    model = DINOv2Detector(backbone_name)
    model.load_state_dict(torch.load(checkpoint_path, map_location=CFG["device"]))
    model.to(CFG["device"])
    model.eval()
    return model

# ── Prédire une image ────────────────────────────────────────────────────────────
@torch.no_grad()
def predict_image(model, image, threshold: float = None):
    """
    Classifie une image unique.
    """
    threshold = threshold or CFG["threshold"]
    model.eval()

    try:
        if isinstance(image, (str, Path)):
            pil = Image.open(image).convert("RGB")
        elif isinstance(image, np.ndarray):
            pil = Image.fromarray((image * 255).astype(np.uint8))
        else:
            pil = image  # déjà PIL
    except Exception as e:
        raise ValueError(f"Erreur lors du chargement de l'image: {e}")

    tensor = transform_224(pil).unsqueeze(0).to(CFG["device"])
    prob = torch.sigmoid(model(tensor)).item()

    return {
        "prediction": 1 if prob >= threshold else 0,
        "probability_real": round(prob, 4),
        "confidence": round(max(prob, 1 - prob), 4),
    }

# ── Prédire un dossier ──────────────────────────────────────────────────────────
@torch.no_grad()
def predict_folder(model, folder: str, batch_size: int = 32) -> list:
    """
    Classifie toutes les images d'un dossier.
    """
    exts = {".jpg", ".jpeg", ".png"}
    paths = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in exts)

    if not paths:
        raise ValueError(f"Aucune image trouvée dans {folder} (extensions supportées: {exts})")

    ds = ImageFolderFlat(folder, transform_224, label=0)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    model.eval()
    all_probs = []
    for x, _ in loader:
        probs = torch.sigmoid(model(x.to(CFG["device"]))).cpu().numpy()
        all_probs.extend(probs)

    results = []
    for path, prob in zip(paths, all_probs):
        results.append({
            "file": str(path),
            "prediction": "REAL" if prob >= CFG["threshold"] else "AI-GENERATED",
            "probability_real": round(float(prob), 4),
            "confidence": round(float(max(prob, 1 - prob)), 4),
        })
    return results

# ── Visualiser les prédictions ─────────────────────────────────────────────────
def show_inference(model, image_paths_or_folder: str or list, true_labels: list = None):
    """
    Visualise les prédictions pour une liste d'images ou un dossier.

    Parameters
    ----------
    image_paths_or_folder : str (chemin vers un dossier) ou list (liste de chemins d'images)
    true_labels : liste de labels réels ("REAL" ou "AI-GENERATED") pour vérifier la prédiction
    """
    # Si c'est un dossier, récupérer tous les chemins d'images
    if isinstance(image_paths_or_folder, str):
        folder_path = Path(image_paths_or_folder)
        exts = {".jpg", ".jpeg", ".png"}
        image_paths = sorted(p for p in folder_path.iterdir() if p.suffix.lower() in exts)
    else:
        image_paths = image_paths_or_folder  # déjà une liste

    n = len(image_paths)
    ncols = min(n, 4)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.array(axes).flatten()
    for ax in axes: ax.axis("off")

    for i, path in enumerate(image_paths):
        pred = predict_image(model, path)
        img = Image.open(path).convert("RGB")
        is_real = pred["prediction"] == 1
        color = "steelblue" if is_real else "firebrick"

        axes[i].imshow(img)
        axes[i].axis("off")
        title = f"{'✓ REAL' if is_real else '✗ AI-GENERATED'}\nP(real)={pred['probability_real']:.3f}"
        if true_labels:
            correct = (pred["prediction"] == 1) == (true_labels[i] == "REAL")
            title += f"  {'✓' if correct else '✗ WRONG'}"
        axes[i].set_title(title, color=color, fontsize=9, fontweight="bold")
        for sp in axes[i].spines.values():
            sp.set_edgecolor(color); sp.set_linewidth(2.5)

    plt.suptitle("Inference Results", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout(); plt.show()





In [3]:
# ── Exemple d'utilisation ───────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Charger le modèle entraîné
    detector = load_trained_model(CFG["backbone"], "/kaggle/input/datasets/kaveeh/test-vae-detector/detector/detector.pth")

    # 2. Prédire une seule image
    result = predict_image(detector, "/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data/COCO_test2014_000000000001.jpg")
    print("Prédiction pour une image:", result)

    # 3. Prédire toutes les images d'un dossier
    results = predict_folder(detector, "/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data")
    print("\nPrédictions pour le dossier:")
    for res in results:
        print(f"{res['file']}: {res['prediction']} (P(real)={res['probability_real']})")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Prédiction pour une image: {'prediction': 1, 'probability_real': 0.7121, 'confidence': 0.7121}

Prédictions pour le dossier:
/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data/COCO_test2014_000000000001.jpg: REAL (P(real)=0.7121)
/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data/COCO_test2014_000000000014.jpg: REAL (P(real)=0.5427)
/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data/COCO_test2014_000000000016.jpg: REAL (P(real)=0.6834)
/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data/COCO_test2014_000000000027.jpg: REAL (P(real)=0.8385)
/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data/COCO_test2014_000000000057.jpg: REAL (P(real)=0.6572)
/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data/COCO_test2014_000000000063.jpg: AI-GENERATED (P(real)=0.0926)
/kaggle/input/datasets/kaveeh/test-vae-detector/test/test/data/COCO_test2014_000000000069.jpg: AI-GENERATED (P(real)=0.4808)
/kaggle/input/datasets/kaveeh/test-vae-d

In [2]:
 # 4. Visualiser les prédictions
show_inference(detector,
    image_paths_or_folder="PLEASE INPUT YOUR OWN PATH",  
    true_labels=None)  

NameError: name 'show_inference' is not defined